## Packages

In [1]:
pip install -q datasets langchain langchain-core langchain-text-splitters sentence-transformers transformers torch faiss-cpu pydantic-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 638.7/638.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import re
import torch
import faiss
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import List
from datasets import load_dataset
from pydantic import BaseModel
from pydantic_ai.models import Model
from pydantic_ai import Agent, RunContext
from pydantic_ai.messages import ModelResponse, TextPart
from langchain_core.documents import Document
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Checks for GPU

In [3]:
!nvidia-smi

Sun Mar 29 15:16:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=55) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [4]:
print("Available GPUs:", torch.cuda.device_count())
if torch.cuda.device_count() > 0:
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU found.")

Available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Dataset Preprocessing

In [5]:
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
train_data = dataset["train"].to_list()
valid_data = dataset["validation"].to_list()
test_data  = dataset["test"].to_list()
df_train = pd.DataFrame(train_data)
df_valid = pd.DataFrame(valid_data)
df_test  = pd.DataFrame(test_data)
print("Train:", df_train.shape, "Validation:", df_valid.shape, "Test:", df_test.shape)
print(df_train.head())

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Train: (287113, 3) Validation: (13368, 3) Test: (11490, 3)
                                             article  \
0  LONDON, England (Reuters) -- Harry Potter star...   
1  Editor's note: In our Behind the Scenes series...   
2  MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...   
3  WASHINGTON (CNN) -- Doctors removed five small...   
4  (CNN)  -- The National Football League has ind...   

                                          highlights  \
0  Harry Potter star Daniel Radcliffe gets £20M f...   
1  Mentally ill inmates in Miami are housed on th...   
2  NEW: "I thought I was going to die," driver sa...   
3  Five small polyps found during procedure; "non...   
4  NEW: NFL chief, Atlanta Falcons owner critical...   

                                         id  
0  42c027e4ff9730fbb3de84c1af0d2c506e41c3e4  
1  ee8871b15c50d0db17b0179a6d2beab35065f1e9  
2  06352019a19ae31e527f37f7571c6dd7f0c5da37  
3  24521a2abb2e1f5e34e6824e0f9e56904a2b0e88  
4  7fe70cc8b12fab2d0a258fababf7d9c6b5

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda")  
print("Model loaded on CUDA:", next(model.parameters()).is_cuda)  

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on CUDA: True


In [7]:
df_train_cpu = df_train

MAX_TOKENS = 512
OVERLAP = 100

def sentence_split(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def split_long_text_by_tokens(text, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    step = max_tokens - overlap

    while start < len(token_ids):
        end = start + max_tokens
        chunk_ids = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)
        start += step

    return chunks

docs = []

for _, row in tqdm(df_train_cpu.iterrows(), total=len(df_train_cpu), desc="Adaptive chunking"):
    article = row["article"]
    highlights = row["highlights"]

    sentences = sentence_split(article)
    if not sentences:
        continue

    # batch tokenization once per article
    sent_token_ids = tokenizer(
        sentences,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    current_ids = []

    for sent, ids in zip(sentences, sent_token_ids):
        sent_len = len(ids)

        # if single sentence itself exceeds limit, flush current chunk first
        if sent_len > MAX_TOKENS:
            if current_ids:
                chunk_text = tokenizer.decode(current_ids, skip_special_tokens=True).strip()
                if chunk_text:
                    docs.append(
                        Document(
                            page_content=chunk_text,
                            metadata={"summary": highlights}
                        )
                    )

            # split oversized sentence directly by tokens
            long_chunks = split_long_text_by_tokens(sent, max_tokens=MAX_TOKENS, overlap=OVERLAP)
            for chunk_text in long_chunks:
                docs.append(
                    Document(
                        page_content=chunk_text,
                        metadata={"summary": highlights}
                    )
                )

            current_ids = []
            continue

        # normal adaptive packing
        if len(current_ids) + sent_len <= MAX_TOKENS:
            current_ids.extend(ids)
        else:
            chunk_text = tokenizer.decode(current_ids, skip_special_tokens=True).strip()
            if chunk_text:
                docs.append(
                    Document(
                        page_content=chunk_text,
                        metadata={"summary": highlights}
                    )
                )

            overlap_ids = current_ids[-OVERLAP:] if OVERLAP > 0 else []
            current_ids = overlap_ids + ids

            # safety in case overlap + sentence exceeds limit
            if len(current_ids) > MAX_TOKENS:
                chunk_text = tokenizer.decode(current_ids[:MAX_TOKENS], skip_special_tokens=True).strip()
                if chunk_text:
                    docs.append(
                        Document(
                            page_content=chunk_text,
                            metadata={"summary": highlights}
                        )
                    )
                current_ids = current_ids[MAX_TOKENS-OVERLAP:] if OVERLAP > 0 else []

    if current_ids:
        chunk_text = tokenizer.decode(current_ids, skip_special_tokens=True).strip()
        if chunk_text:
            docs.append(
                Document(
                    page_content=chunk_text,
                    metadata={"summary": highlights}
                )
            )

print(docs[:3])
print("Total chunks:", len(docs))

Adaptive chunking: 100%|██████████| 287113/287113 [18:34<00:00, 257.66it/s]

[Document(metadata={'summary': "Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .\nYoung actor says he has no plans to fritter his cash away .\nRadcliffe's earnings from first five Potter films have been held in trust fund ."}, page_content='london, england ( reuters ) - - harry potter star daniel radcliffe gains access to a reported £20 million ( $ 41. 1 million ) fortune as he turns 18 on monday, but he insists the money won \' t cast a spell on him. daniel radcliffe as harry potter in " harry potter and the order of the phoenix " to the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. " i don \' t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar, " he told an australian interviewer earlier this month. " i don \' t think i \' ll be particularly extravagant. " th

In [8]:
seen = set()
unique_docs = []

for doc in docs:
    clean_text = doc.page_content.strip()
    if clean_text not in seen:
        seen.add(clean_text)
        unique_docs.append(doc)
df_docs = pd.DataFrame({
    "text": [doc.page_content for doc in unique_docs],
    "summary": [doc.metadata["summary"] for doc in unique_docs]
})

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device = "cuda" if torch.cuda.is_available() else "cpu"
)


# %% 2. Prepare your text corpus (list of strings)
texts = df_docs["text"].tolist()

# %% 3. Compute embeddings in batches (float32) WITH PROGRESS BAR
def compute_embeddings(text_list, batch_size=256):
    all_emb = []
    for i in tqdm(range(0, len(text_list), batch_size), desc="Encoding Embeddings"):
        batch = text_list[i:i + batch_size]
        embs = embedding_model.encode(
            batch,
            batch_size=len(batch),
            convert_to_numpy=True,
            device=embedding_model.device
        )
        all_emb.append(embs.astype(np.float32))
    return np.vstack(all_emb)

embeddings_np = compute_embeddings(texts, batch_size=256)
embedding_dim = embeddings_np.shape[1]
print("Computed embeddings shape:", embeddings_np.shape, "dim =", embedding_dim)

# %% 4. Build compressed FAISS index: IVF + PQ
nlist = 512
m = 64
nbits = 8

quantizer = faiss.IndexFlatL2(embedding_dim)
index = faiss.IndexIVFPQ(quantizer, embedding_dim, nlist, m, nbits)
index.nprobe = 32

# ---- TRAINING WITH PROGRESS BAR ----
# FAISS train() accepts data in 1 shot, so we simulate progress by chunking manually.
def train_with_progress(index, data, chunk_size=10000):
    print("Training quantizer...")
    # FAISS wants a single array at the end, but we can show progress filling a buffer.
    training_data = []
    for i in tqdm(range(0, len(data), chunk_size), desc="Preparing Training Data"):
        training_data.append(data[i:i + chunk_size])
    training_data = np.vstack(training_data)
    
    index.train(training_data)   # actual FAISS training (fast after data is assembled)
    print("Training complete.")

train_with_progress(index, embeddings_np)

# ---- ADD embeddings in batches WITH PROGRESS BAR ----
def add_with_progress(index, data, batch_size=10000):
    print("Adding embeddings to index...")
    for i in tqdm(range(0, len(data), batch_size), desc="FAISS Index Add"):
        index.add(data[i:i + batch_size])

add_with_progress(index, embeddings_np)

# %% 5. Save index
faiss.write_index(index, "/kaggle/working/cnn_dailymail_ivfpq.index")
print("Saved IVF-PQ index.")

## Document Retrieval

In [9]:
import faiss

INDEX_PATH = "/kaggle/input/datasets/vishnu0107/llm-agentic-rag-dataset/cnn_dailymail_ivfpq.index"

# Load index
index = faiss.read_index(INDEX_PATH)

print("Index loaded successfully")
print("Total vectors:", index.ntotal)

Index loaded successfully
Total vectors: 686735


In [10]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device = "cuda" if torch.cuda.is_available() else "cpu"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# --- Configuration: set your index file path here ---
#INDEX_PATH = "/kaggle/input/ivfpq-cnn-set/cnn_dailymail_ivfpq.index"
# Load FAISS index
#index = faiss.read_index(INDEX_PATH)
# def retrieve_documents(query, index, k=10, final_k=5):
#     """
#     Retrieves top-k documents using FAISS and reranks them using cosine similarity.
#     """
#     query_embedding = embedding_model.encode(query).reshape(1, -1).astype("float32")
#     distances, indices = index.search(query_embedding, k)

#     # Get the candidate documents
#     candidate_docs = [df_docs["text"].iloc[idx] for idx in indices[0] if idx < len(df_docs)]

#     # Rerank by cosine similarity
#     reranked = sorted(
#         candidate_docs,
#         key=lambda doc: cosine_similarity(
#             [embedding_model.encode(query)],
#             [embedding_model.encode(doc)]
#         )[0][0],
#         reverse=True
#     )

#     return reranked[:final_k]

def retrieve_documents(query, index, k=10, final_k=5):
    # Encode query ONCE
    query_embedding = embedding_model.encode(query, normalize_embeddings=True)
    
    distances, indices = index.search(query_embedding.reshape(1, -1).astype("float32"), k)

    candidate_docs = [df_docs["text"].iloc[idx] for idx in indices[0] if idx < len(df_docs)]

    # Batch encode docs
    doc_embeddings = embedding_model.encode(candidate_docs, normalize_embeddings=True)

    # Fast cosine (dot product since normalized)
    scores = doc_embeddings @ query_embedding

    # Rerank
    ranked_docs = [doc for _, doc in sorted(zip(scores, candidate_docs), reverse=True)]

    return ranked_docs[:final_k]

## Loading Models

In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("Graph_RAG")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
llama3_model_name = "meta-llama/Llama-3.2-3B-Instruct"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, token=hf_token)
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    token=hf_token,
    torch_dtype=torch.float16,  
    device_map="auto" 
)

print("Meta-Llama 3.2 3B successfully loaded with 4-bit quantization!")

In [ ]:
del llama3_model

In [ ]:
torch.cuda.empty_cache()

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
llama3_model_name = "Qwen/Qwen2.5-7B-Instruct"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, token=hf_token)
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    token=hf_token,
    torch_dtype=torch.float16,  
    device_map="auto" 
)

print("Qwen2.5-7B-Instruct CNN 12-6 successfully loaded with 4-bit quantization!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B-Instruct CNN 12-6 successfully loaded with 4-bit quantization!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
llama3_model_name = "sshleifer/distilbart-cnn-12-6"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, token=hf_token)
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    token=hf_token,
    torch_dtype=torch.float16,  
    device_map="auto" 
)

print("Sshleifer's Distilbart CNN 12-6 successfully loaded with 4-bit quantization!")

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
llama3_model_name = "microsoft/Phi-3.5-mini-instruct"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, token=hf_token)
llama3_model = AutoModelForCausalLM.from_pretrained(
    llama3_model_name,
    token=hf_token,
    torch_dtype=torch.float16,  
    device_map="auto" 
)

print("Phi-3.5-mini-instruct successfully loaded with 4-bit quantization!")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

Phi-3.5-mini-instruct successfully loaded with 4-bit quantization!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
llama3_model_name = "google/flan-t5-large"
llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_model_name, token=hf_token)
llama3_model = AutoModelForSeq2SeqLM.from_pretrained(
    llama3_model_name,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Flan-t5-large successfully loaded with 4-bit quantization!")

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Flan-t5-large successfully loaded with 4-bit quantization!


In [30]:
llama3_model.config.pad_token_id = llama3_model.config.eos_token_id

## Repsonse Generation

In [31]:
# For general models
def generate_response(query, retriever, generator, tokenizer, index):
    # Retrieve relevant documents
    #retrieved_chunks = retrieve_documents(query, index)
    retrieved_chunks = retrieve_documents(
    query,
    index=index,
    final_k=5
    )
    print("\n Retrieved Chunks:")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"\n Chunk {i+1}:\n{chunk}\n")
    # Truncate and prepare context
    context_tokens = tokenizer.encode(" ".join(retrieved_chunks), truncation=True, max_length=2048)
    context = tokenizer.decode(context_tokens, skip_special_tokens=True)
    # Explicit prompt formatting to force grounding
    input_text = f"""
You are a helpful assistant. Only answer using the context provided.

### Context:
{context}

### Question:
{query}

### Answer:
""".strip()

    # Tokenize and move to device
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    # Generate response
    output = generator.generate(**inputs, max_new_tokens=256)

    # Decode and return
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response

In [16]:
#for bart models
def generate_response(query, retriever, generator, tokenizer, index):
    # 1. Retrieve exactly like you did before
    retrieved_chunks = retrieve_documents(
        query,
        index=index,
        final_k=5
    )
    
    context = "\n\n".join(retrieved_chunks)

    # 2. Llama-3 Prompt Template
    # Llama-3 works best with this specific "Instruct" format
    input_text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" \
                 f"You are a helpful assistant. Only answer using the context provided.<|eot_id|>" \
                 f"<|start_header_id|>user<|end_header_id|>\n\n" \
                 f"Context:\n{context}\n\nQuestion: {query}<|eot_id|>" \
                 f"<|start_header_id|>assistant<|end_header_id|>\n\n"

    # 3. Tokenize and move to GPU
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    # 4. Generate
    # We use max_new_tokens so it doesn't matter how long the context is
    output = generator.generate(
        **inputs, 
        max_new_tokens=256, 
        temperature=0.1, 
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )

    # 5. Decode ONLY the new tokens
    # Causal models (Llama) return the input + output. We slice it to get only the answer.
    prompt_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)
    
    return response.strip()

In [32]:
query = "What is the name of the actor of Harry Potter? Answer from the embedings"
llama3_model.config.pad_token_id = llama3_model.config.eos_token_id   # Set pad token ID explicitly
llama3_tokenizer.pad_token = llama3_tokenizer.eos_token
response = generate_response(query, retrieve_documents, llama3_model, llama3_tokenizer, index)
print(response)


 Retrieved Chunks:

 Chunk 1:
( emma watson ), the bookish and indispensible member of their clan, who has demonstrated key leadership qualities ; and all the rest, preparing for the showdown with the archvillain voldemort. watch the potter cast answer your questions ». among the returning performers : michael gambon as dumbledore, alan rickman as snape, maggie smith as mcgonagall, robbie coltrane as hagrid and ralph fiennes as voldemort. one addition is jim broadbent, who plays horace slughorn, a former professor brought back to hogwarts. watch the stars at the rainy premiere ». despite the growing darkness, there ' s also a lightness to the new film, says daniel radcliffe, who plays harry. unlike the previous two movies in the series, which were rated pg - 13, this one is rated a more family - friendly pg. " i think this film ' s funnier, " radcliffe said. " there are a couple of moments which i laughed out loud at. " not that it ' s going to be a barrel of laughs, yates cautions. (

## Agentic RAG using Pydantic AI

In [19]:
class HuggingFaceModel(Model):

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @property
    def model_name(self):
        return "hf-llama3-local"

    @property
    def system(self):
        return "huggingface"

    async def request(
        self,
        messages,
        model_settings=None,
        model_request_parameters=None
    ):
        # ---- BUILD PROMPT ----
        prompt_parts = []

        for m in messages:
            for p in m.parts:
                if hasattr(p, "content"):
                    prompt_parts.append(p.content)

        prompt = "\n".join(prompt_parts)

        # ---- TOKENIZE ----
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True
        ).to("cuda")

        # ---- GENERATE ----
        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.0,
                do_sample=False
            )

        text = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return ModelResponse(parts=[TextPart(content=text)])

In [20]:
hf_llm = HuggingFaceModel(llama3_model, llama3_tokenizer)

In [21]:
rag_agent = Agent(
    model=hf_llm,
    system_prompt="""
You are a RAG assistant.

Workflow:
1. Always call retrieve_context tool first
2. Use ONLY retrieved chunks to answer
3. If answer not found, say 'Not enough information'

Return only the final answer.
"""
)

In [22]:
class RetrievalOutput(BaseModel):
    chunks: List[str]

@rag_agent.tool
def retrieve_context(ctx: RunContext, query: str) -> str:
    chunks = retrieve_documents(query, index)

    return "\n\n".join(chunks)

In [23]:
def clean_output(text):
    if "Answer:" in text:
        text = text.split("Answer:")[-1]

    text = re.split(r"\n\s*(What|Question)", text)[0]

    return text.strip()

# result = await rag_agent.run(
#     "Who played Harry Potter?"
# )

# print(result.output)
result = await rag_agent.run("Who played Harry Potter?")
print(clean_output(result.output))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You are a RAG assistant.

Workflow:
1. Always call retrieve_context tool first
2. Use ONLY retrieved chunks to answer
3. If answer not found, say 'Not enough information'

Return only the final answer.

Who played Harry Potter? Not enough information


In [17]:
!pip install rouge-score nltk evaluate bert_score sentence-transformers

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=55) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00


In [18]:
import pandas as pd
from tqdm import tqdm
import evaluate
from sentence_transformers import SentenceTransformer, util

In [19]:
!pip install evaluate rouge-score bert_score

In [20]:
def retrieve_batch(queries, index, k=20, final_k=8):
    query_embeddings = embedding_model.encode(queries, normalize_embeddings=True)

    distances, indices = index.search(query_embeddings.astype("float32"), k)

    results=[]

    for i, query in enumerate(queries):
        candidate_docs = [df_docs["text"].iloc[idx] for idx in indices[i] if idx < len(df_docs)]

        doc_embeddings = embedding_model.encode(candidate_docs, normalize_embeddings=True)
        scores = doc_embeddings @ query_embeddings[i]

        ranked_docs = [doc for _, doc in sorted(zip(scores, candidate_docs), reverse=True)]
        results.append(ranked_docs[:final_k])

    return results

In [21]:
def generate_batch(queries, contexts, tokenizer, model):
    inputs = [
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"You are a helpful assistant. Only answer using the context provided.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"Context:\n{ctx}\n\nQuestion: {q}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
        for q, ctx in zip(queries, contexts)
    ]

    tokenized = tokenizer(
        inputs,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **tokenized,
            max_new_tokens=96,   # ↓ BIG speed gain
            do_sample=False
        )

    responses = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    return responses

In [22]:
import numpy as np
import pandas as pd
import evaluate
from tqdm import tqdm

# ---- SMART TRUNCATION ----
def smart_truncate(text, head=700, tail=300):
    if len(text) <= head + tail:
        return text
    return text[:head] + " ... " + text[-tail:]


def evaluate_model_fast(df_test, num_samples=1000, batch_size=16):
    
    # ---- LOAD METRICS ----
    rouge_metric = evaluate.load('rouge')
    bleu_metric = evaluate.load('bleu')
    bertscore_metric = evaluate.load('bertscore')

    # ---- SAMPLE DATA ----
    test_subset = df_test.sample(n=num_samples, random_state=42).reset_index(drop=True)

    print(f"Starting evaluation on {num_samples} samples...")

    all_generated = []
    all_references = []

    # ---- BATCH LOOP ----
    for i in tqdm(range(0, num_samples, batch_size)):
        batch = test_subset.iloc[i:i+batch_size]

        # 🔹 BETTER QUERY FORMULATION
        queries = [
            f"""Generate a concise CNN/DailyMail-style news summary:
- Capture key events
- Mention important entities
- Focus on outcomes
- Keep it within 2-3 sentences

Article:
{smart_truncate(row['article'])}
"""
            for _, row in batch.iterrows()
        ]

        references = [row['highlights'] for _, row in batch.iterrows()]

        try:
            # 🔹 RETRIEVAL
            retrieved_docs = retrieve_batch(queries, index)

            # 🔹 CLEAN CONTEXT (reduce noise)
            contexts = [
                "\n\n".join([doc[:300] for doc in docs])
                for docs in retrieved_docs
            ]

            # 🔹 GENERATION
            generated_answers = generate_batch(
                queries,
                contexts,
                llama3_tokenizer,
                llama3_model
            )

            # 🔹 CLEAN OUTPUT
            cleaned = []
            for g in generated_answers:
                g = g.split("### Answer:")[-1]
                g = g.strip()
                cleaned.append(g)

            all_generated.extend(cleaned)
            all_references.extend(references)

        except Exception as e:
            print(f"Batch error: {e}")
            continue

    # ---- CREATE DATAFRAME ----
    results_df = pd.DataFrame({
        "generated_answer": all_generated,
        "reference": all_references
    })

    print("Computing semantic similarity (batched)...")

    # 🔹 FAST EMBEDDINGS
    gen_emb = similarity_model.encode(
        results_df['generated_answer'].tolist(),
        batch_size=64,
        normalize_embeddings=True
    )

    ref_emb = similarity_model.encode(
        results_df['reference'].tolist(),
        batch_size=64,
        normalize_embeddings=True
    )

    # 🔹 COSINE SIM (FAST DOT PRODUCT)
    cosine_sim = np.sum(gen_emb * ref_emb, axis=1)
    results_df["semantic_similarity"] = cosine_sim

    print("Calculating aggregate scores...")

    # ---- METRICS ----
    rouge_results = rouge_metric.compute(
        predictions=results_df['generated_answer'].tolist(),
        references=results_df['reference'].tolist()
    )

    bleu_results = bleu_metric.compute(
        predictions=results_df['generated_answer'].tolist(),
        references=[[r] for r in results_df['reference'].tolist()]
    )

    bert_results = bertscore_metric.compute(
        predictions=results_df['generated_answer'].tolist(),
        references=results_df['reference'].tolist(),
        lang="en"
    )

    # ---- FINAL METRICS ----
    final_metrics = {
        "ROUGE-1": float(rouge_results['rouge1']),
        "ROUGE-2": float(rouge_results['rouge2']),
        "ROUGE-L": float(rouge_results['rougeL']),
        "BLEU": float(bleu_results['bleu']),
        "BERTScore_F1": float(np.mean(bert_results['f1'])),
        "Avg_Semantic_Similarity": float(results_df['semantic_similarity'].mean())
    }

    # ---- SAVE RESULTS ----
    results_df.to_csv("detailed_eval_results.csv", index=False)
    pd.DataFrame([final_metrics]).to_csv("summary_metrics.csv", index=False)

    print("\n--- Evaluation Complete ---")
    for k, v in final_metrics.items():
        print(f"{k}: {v:.4f}")

    return final_metrics

In [23]:
from sentence_transformers import SentenceTransformer

similarity_model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    device='cuda'   # use GPU (critical for speed)
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
evaluate_model_fast(df_test, num_samples=200, batch_size=16)

Starting evaluation on 200 samples...


100%|██████████| 13/13 [01:18<00:00,  6.07s/it]


Computing semantic similarity (batched)...
Calculating aggregate scores...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Evaluation Complete ---
ROUGE-1: 0.0476
ROUGE-2: 0.0042
ROUGE-L: 0.0398
BLEU: 0.0000
BERTScore_F1: 0.7904
Avg_Semantic_Similarity: 0.1712


{'ROUGE-1': 0.04762050068937847,
 'ROUGE-2': 0.004227779439764963,
 'ROUGE-L': 0.03979735832269127,
 'BLEU': 0.0,
 'BERTScore_F1': 0.79035809725523,
 'Avg_Semantic_Similarity': 0.17118023335933685}

In [ ]:
def generate_response(query, retriever, generator, tokenizer, index):
    # 1. Retrieve exactly like you did before
    retrieved_chunks = retrieve_documents(
        query,
        index=index,
        final_k=5
    )
    
    context = "\n\n".join(retrieved_chunks)

    # 2. Llama-3 Prompt Template
    # Llama-3 works best with this specific "Instruct" format
    input_text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n" \
                 f"You are a helpful assistant. Only answer using the context provided.<|eot_id|>" \
                 f"<|start_header_id|>user<|end_header_id|>\n\n" \
                 f"Context:\n{context}\n\nQuestion: {query}<|eot_id|>" \
                 f"<|start_header_id|>assistant<|end_header_id|>\n\n"

    # 3. Tokenize and move to GPU
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    # 4. Generate
    # We use max_new_tokens so it doesn't matter how long the context is
    output = generator.generate(
        **inputs, 
        max_new_tokens=256, 
        temperature=0.1, 
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )

    # 5. Decode ONLY the new tokens
    # Causal models (Llama) return the input + output. We slice it to get only the answer.
    prompt_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)
    
    return response.strip()

In [ ]:
import pandas as pd
from tqdm import tqdm
import evaluate
from sentence_transformers import SentenceTransformer, util

# Load evaluation metrics
rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('bleu')
bertscore_metric = evaluate.load('bertscore')
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

def evaluate_model(df_test, num_samples=50):
    """
    Evaluates the model on a subset of test data and saves results to CSV.
    """
    results = []
    
    # Selecting a subset for evaluation to save time/compute in Kaggle
    test_subset = df_test.sample(n=num_samples, random_state=42)
    
    print(f"Starting evaluation on {num_samples} samples...")
    
    for _, row in tqdm(test_subset.iterrows(), total=num_samples):
        query = f"Summarize the following article: {row['article'][:500]}..." # Example query
        reference = row['highlights']
        
        try:
            # Generate response using your specific function
            generated_answer = generate_response(query, retrieve_documents, llama3_model, llama3_tokenizer, index)
            # Clean response to remove prompt artifacts
            generated_answer = generated_answer.split("### Answer:")[-1].strip()
            
            # 1. Semantic Similarity
            emb1 = similarity_model.encode(generated_answer, convert_to_tensor=True)
            emb2 = similarity_model.encode(reference, convert_to_tensor=True)
            cosine_sim = util.pytorch_cos_sim(emb1, emb2).item()
            
            results.append({
                "article": row['article'][:200],
                "reference": reference,
                "generated_answer": generated_answer,
                "semantic_similarity": cosine_sim
            })
        except Exception as e:
            print(f"Error processing sample: {e}")
            continue

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # 2. Calculate Aggregate Metrics
    print("Calculating aggregate scores...")
    
    # ROUGE
    rouge_results = rouge_metric.compute(
        predictions=results_df['generated_answer'], 
        references=results_df['reference']
    )
    
    # BLEU
    bleu_results = bleu_metric.compute(
        predictions=results_df['generated_answer'], 
        references=[[r] for r in results_df['reference']]
    )
    
    # BERTScore (using RoBERTa-large as is standard for papers)
    bert_results = bertscore_metric.compute(
        predictions=results_df['generated_answer'], 
        references=results_df['reference'], 
        lang="en"
    )
    
    # Summary Statistics
    final_metrics = {
        "ROUGE-1": rouge_results['rouge1'],
        "ROUGE-2": rouge_results['rouge2'],
        "ROUGE-L": rouge_results['rougeL'],
        "BLEU": bleu_results['bleu'],
        "BERTScore_F1": sum(bert_results['f1']) / len(bert_results['f1']),
        "Avg_Semantic_Similarity": results_df['semantic_similarity'].mean()
    }
    
    # Save detailed results
    results_df.to_csv("detailed_eval_results.csv", index=False)
    
    # Save summary metrics
    pd.DataFrame([final_metrics]).to_csv("summary_metrics.csv", index=False)
    
    print("\n--- Evaluation Complete ---")
    for k, v in final_metrics.items():
        print(f"{k}: {v:.4f}")
    
    return final_metrics

# Execute Evaluation
metrics = evaluate_model(df_test, num_samples=1000)